In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler,LabelEncoder 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [3]:
df = pd.read_csv('used_cars.csv')

In [5]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [18]:
df['accident'].unique()

array(['At least 1 accident or damage reported', 'None reported', nan],
      dtype=object)

In [8]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         4009 non-null   object
 1   model         4009 non-null   object
 2   model_year    4009 non-null   int64 
 3   milage        4009 non-null   object
 4   fuel_type     3839 non-null   object
 5   engine        4009 non-null   object
 6   transmission  4009 non-null   object
 7   ext_col       4009 non-null   object
 8   int_col       4009 non-null   object
 9   accident      3896 non-null   object
 10  clean_title   3413 non-null   object
 11  price         4009 non-null   object
dtypes: int64(1), object(11)
memory usage: 376.0+ KB
None


In [20]:
# Strip "$" and "," from price -> float
df['price_num'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
 
# Strip "mi." and "," from mileage -> float
df['mileage_num'] = df['milage'].replace(r'[a-zA-Z. ,]', '', regex=True).astype(float)
 
# Drop extreme outliers (ultra-luxury cars skew price stats/model heavily)
df_clean = df[df['price_num'] < 400_000].copy()

In [22]:
print("Dataset shape:", df.shape)
print(df_clean[['price_num', 'mileage_num', 'model_year']].describe())


Dataset shape: (4009, 14)
           price_num    mileage_num   model_year
count    3990.000000    3990.000000  3990.000000
mean    41035.996992   64998.656642  2015.511278
std     40171.310546   52259.544601     6.106127
min      2000.000000     100.000000  1974.000000
25%     17000.000000   23294.000000  2012.000000
50%     31000.000000   53134.500000  2017.000000
75%     49663.500000   94411.000000  2020.000000
max    399950.000000  405000.000000  2024.000000


In [23]:
top_brands = df_clean['brand'].value_counts().head(10).index
print("\nTop 10 brands by listing count:")
print(df_clean['brand'].value_counts().head(10))


Top 10 brands by listing count:
brand
Ford             385
BMW              375
Mercedes-Benz    314
Chevrolet        292
Audi             200
Toyota           199
Porsche          198
Lexus            163
Jeep             143
Land             130
Name: count, dtype: int64


In [24]:
print("\nAverage price by brand (top 10 by volume):")
print(
    df_clean[df_clean['brand'].isin(top_brands)]
    .groupby('brand')['price_num']
    .mean()
    .sort_values(ascending=False)
)


Average price by brand (top 10 by volume):
brand
Porsche          77441.974747
Land             55764.061538
Mercedes-Benz    50888.108280
BMW              41072.309333
Audi             39907.430000
Chevrolet        36722.739726
Lexus            35668.521472
Ford             35218.135065
Jeep             31099.790210
Toyota           30026.000000
Name: price_num, dtype: float64


In [25]:
print("\nAverage price: accident history vs none:")
print(df_clean.groupby('accident')['price_num'].mean())


Average price: accident history vs none:
accident
At least 1 accident or damage reported    25861.701523
None reported                             45823.224066
Name: price_num, dtype: float64


In [26]:
print("\nCorrelations with price:")
print("  mileage:", df_clean['mileage_num'].corr(df_clean['price_num']))
print("  model_year:", df_clean['model_year'].corr(df_clean['price_num']))


Correlations with price:
  mileage: -0.5101504423870865
  model_year: 0.42578083743465633
